## Analyze the Distribution of Delays

In [ ]:
%load_ext autoreload
%autoreload 2

import pylupnt as pnt
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

In [ ]:
figdir = os.path.join(pnt.get_output_dir(), "iono_delay", "gnss_plots")

In [ ]:
# Load ray tracing pickele file
def load_pickle_file(sim_params, savedir):
    ymdh = sim_params["epoch_ymdh"]
    lcrns_idx = sim_params["lcrns_idx"]
    signal_family = sim_params["signal_family"]
    rz12 = sim_params["rz12"]
    gnss_const = sim_params["gnss_const"]
    kp = sim_params.get("kp", None)
    n_orbit = sim_params.get("n_orbit", 3)
    dt = sim_params.get("dt", 10)
    dtrt = sim_params.get("dt_raytrace", 60)

    orbit_dt_str = "norbit_{}_dt_{}s_dtrt_{}s".format(int(n_orbit), int(dt), int(dtrt))

    epoch_dict = {
        "year": ymdh[0],
        "month": ymdh[1],
        "day": ymdh[2],
        "hour": ymdh[3],
        "minute": 0,
        "second": 0,
    }
    epoch_str = (
        "{year}_{month:02d}_{day:02d}_{hour:02d}_{minute:02d}_{second:02d}".format(
            **epoch_dict
        )
    )

    outfilename = (
        "ionodata_raytrace_{}_sat_{}_{}_signal_{}_rz12_{:.1f}_kp_{:.1f}_pco.pkl".format(
            epoch_str, lcrns_idx, gnss_const, signal_family, rz12, kp
        )
    )

    savedir = os.path.join(savedir, orbit_dt_str)
    updated_pickle_filename = os.path.join(savedir, outfilename)

    if os.path.exists(updated_pickle_filename):
        print(f"Loading {outfilename}")
        df = pd.read_pickle(updated_pickle_filename)
    else:
        print(f"{outfilename} not found.")
        df = None

    return df

In [ ]:
# Load L1, L5 signal for each constellation
savedir = os.path.join(pnt.get_output_dir(), "iono_delay", "raytrace_summary_pco")

df_signals = {}
for gnss_const in ["GPS", "GALILEO", "QZSS"]:
    df_signals[gnss_const] = {}
    for signal_family in [1, 5]:
        sim_params = {
            "gnss_const": gnss_const,
            "signal_family": signal_family,
            "lcrns_idx": 0,
            "epoch_ymdh": [2025, 3, 1, 12],
            "rz12": 50.0,  # -1.0 for historical or projected R12
            "kp": 3.0,
            "n_orbit": 6,
            "dt": 1,  # time step in seconds
            "dt_raytrace": 120,  # raytracing time step in seconds
        }
        df_signals[gnss_const][signal_family] = load_pickle_file(sim_params, savedir)

## Plot Ephemeris Errors

In [ ]:
from src.plots_ionodata import plot_ephemeris_error

if not os.path.exists(figdir):
    os.makedirs(figdir, exist_ok=True)
filename = os.path.join(figdir, "ephemeris_error_histogram.pdf")
plot_ephemeris_error(df_signals, diff_norm=True, inv=1, filename=filename)

In [ ]:
print(df_signals["GPS"][1].keys())  # Example to show the loaded DataFrame

## Generate TDCP Errors

In [ ]:
from src.plots_ionodata import generate_timestep_df

timestep_df_gps = generate_timestep_df(df_signals, "GPS", [1, 5], tidx_inv=10)
timestep_df_galileo = generate_timestep_df(df_signals, "GALILEO", [1, 5], tidx_inv=10)
timestep_df_qzss = generate_timestep_df(df_signals, "QZSS", [1, 5], tidx_inv=10)

timestep_df = {
    "GPS": timestep_df_gps,
    "GALILEO": timestep_df_galileo,
    "QZSS": timestep_df_qzss,
}

In [ ]:
# save to pickle file (in figdir)
pickle_filename = os.path.join(figdir, "timestep_df.pkl")

import pickle

with open(pickle_filename, "wb") as f:
    pickle.dump(timestep_df, f)

print(f"Saved timestep tracked sats to {pickle_filename}")

## Load Pickle Files

In [ ]:
# Load dataframe
import pickle

pickle_filename = os.path.join(figdir, "timestep_df.pkl")
with open(pickle_filename, "rb") as f:
    timestep_df = pickle.load(f)

In [ ]:
from src.plots_ionodata import plot_tracked_sats

filename = os.path.join(figdir, "tracked_sats_over_time.pdf")
plot_tracked_sats(
    timestep_df, gnss_consts=["GPS", "GALILEO", "QZSS"], filename=filename
)

## TDCP errors

In [ ]:
from src.plots_ionodata import plot_tdcp_errors

filename = os.path.join(figdir, "tdcp_error_profiles.pdf")
plot_tdcp_errors(
    timestep_df,
    gnss_consts=["GPS", "GALILEO", "QZSS"],
    signal_family=1,
    filename=filename,
)

## URE Errors for TDCP

In [ ]:
from src.plots_ionodata import plot_iono_free_ure_errors

filename = os.path.join(figdir, "iono_free_ure_error_profiles.pdf")
plot_iono_free_ure_errors(
    timestep_df, gnss_consts=["GPS", "GALILEO", "QZSS"], plot_inv=10, filename=filename
)

## Attitude vs Ionofree

In [ ]:
from src.plots_ionodata import plot_iono_delays_altitude

filename = os.path.join(figdir, "iono_delay_altitude.pdf")
plot_iono_delays_altitude(timestep_df, plot_inv=60, filename=filename)

## Altitude vs Receiver Noise

In [ ]:
from src.plots_ionodata import plot_noise_altitude

filename = os.path.join(figdir, "noise_altitude.pdf")
plot_noise_altitude(timestep_df, plot_inv=60, filename=filename)